# Comorbidity Grouping: Organ-System Categories

Derives the twelve clinically defined comorbidity categories used throughout this study's feature representation 

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

## 1. Load baseline comorbidity flags

In [2]:
df = pd.read_csv('baseline_features.csv')
comorbid_cols = sorted([c for c in df.columns if c.startswith('comorbid_')])
print(f'{len(df):,} patients, {len(comorbid_cols)} comorbidity flags')

prevalence = df[comorbid_cols].mean().sort_values(ascending=False)
print()
print('Prevalence (%):')
print((prevalence * 100).round(1))

3,050 patients, 22 comorbidity flags

Prevalence (%):
comorbid_Osteo           5.6
comorbid_Cancer          4.6
comorbid_Obesity         3.3
comorbid_BackPain        2.3
comorbid_FluidsLytes     2.2
comorbid_CKD             1.1
comorbid_Liver           0.8
comorbid_WeightLoss      0.8
comorbid_Anemia          0.7
comorbid_Asthma          0.7
comorbid_MHC             0.6
comorbid_NeuroOther      0.6
comorbid_COPD            0.6
comorbid_Paralysis       0.5
comorbid_Arthritis       0.3
comorbid_Alcohol         0.3
comorbid_Coagulopathy    0.3
comorbid_PUD             0.2
comorbid_Gout            0.2
comorbid_Hypothyroid     0.2
comorbid_Drugs           0.1
comorbid_BloodLoss       0.1
dtype: float64


## 2. Organ-system category counts (clinical grouping)

Author-derived grouping of the 22 Elixhauser-style flags into 12 clinically coherent categories.

In [3]:
CLINICAL_CATEGORIES = {
    'cat_Renal': ['comorbid_CKD'],
    'cat_FluidElectrolyte': ['comorbid_FluidsLytes'],
    'cat_Metabolic': ['comorbid_Obesity', 'comorbid_Gout'],
    'cat_Respiratory': ['comorbid_COPD', 'comorbid_Asthma'],
    'cat_MentalHealthSubstance': ['comorbid_Alcohol', 'comorbid_Drugs', 'comorbid_MHC'],
    'cat_Musculoskeletal': ['comorbid_Arthritis', 'comorbid_BackPain', 'comorbid_Osteo'],
    'cat_Hematologic': ['comorbid_Anemia', 'comorbid_BloodLoss', 'comorbid_Coagulopathy'],
    'cat_Hepatic': ['comorbid_Liver', 'comorbid_PUD'],
    'cat_Neurologic': ['comorbid_NeuroOther', 'comorbid_Paralysis'],
    'cat_Oncologic': ['comorbid_Cancer'],
    'cat_EndocrineOther': ['comorbid_Hypothyroid'],
    'cat_ConstitutionalFrailty': ['comorbid_WeightLoss'],
}

assigned = [c for cats in CLINICAL_CATEGORIES.values() for c in cats]
assert sorted(assigned) == sorted(comorbid_cols), f'mismatch: {set(comorbid_cols) ^ set(assigned)}'
assert len(assigned) == len(set(assigned)), 'a flag was assigned to more than one category'
print(f'All {len(comorbid_cols)} flags assigned to exactly one of {len(CLINICAL_CATEGORIES)} categories -- verified.')

clinical_category_features = pd.DataFrame({
    cat_name: df[cols].sum(axis=1) for cat_name, cols in CLINICAL_CATEGORIES.items()
})
clinical_category_features.describe().T[['mean', 'std', 'min', 'max']]

All 22 flags assigned to exactly one of 12 categories -- verified.


,mean,std,min,max
cat_Renal,0.010820,0.103470,0.0,1.0
cat_FluidElectrolyte,0.021967,0.146600,0.0,1.0
cat_Metabolic,0.034754,0.183186,0.0,1.0
cat_Respiratory,0.012787,0.112372,0.0,1.0
cat_MentalHealthSubstance,0.010492,0.111144,0.0,2.0
cat_Musculoskeletal,0.081639,0.278609,0.0,2.0
cat_Hematologic,0.010820,0.103470,0.0,1.0
cat_Hepatic,0.010164,0.100319,0.0,1.0
cat_Neurologic,0.011148,0.105009,0.0,1.0
cat_Oncologic,0.045574,0.208593,0.0,1.0


## 3. Save combined feature file

In [4]:
combined = pd.concat([
    df[['person_id']],
    clinical_category_features,
], axis=1)

combined.to_csv('comorbidity_grouping_features.csv', index=False)
print(f'Saved comorbidity_grouping_features.csv: {combined.shape[0]:,} patients x {combined.shape[1]-1} derived features ({len(CLINICAL_CATEGORIES)} organ-system categories)')
combined.head()


Saved comorbidity_grouping_features.csv: 3,050 patients x 12 derived features (12 organ-system categories)


,person_id,cat_Renal,cat_FluidElectrolyte,cat_Metabolic,cat_Respiratory,cat_MentalHealthSubstance,cat_Musculoskeletal,cat_Hematologic,cat_Hepatic,cat_Neurologic,cat_Oncologic,cat_EndocrineOther,cat_ConstitutionalFrailty
0,438,0,0,0,0,0,0,0,0,0,0,0,0
1,683,0,0,0,0,0,0,0,0,0,0,0,0
2,686,0,0,0,1,0,0,0,0,0,1,0,0
3,939,0,0,0,0,0,0,0,0,0,0,0,0
4,951,0,0,0,0,0,0,0,0,0,0,0,0
